# PHI Guard v7: demo

One byte-level model, four answers per clinical note, CPU only:

| output | question | use on |
|---|---|---|
| `score` | Is PHI left in this de-identified note? | de-identified text |
| `harmful_score` | Is that leak harmful? (never above `score`) | de-identified text |
| `level` 0-6 (`score_1`..`score_6`) | How much PHI does the note expose? | raw or de-identified text |
| `raw_score` | Does this raw note contain PHI at all? | raw text, before de-identification |

Two models with the same outputs: **phi-guard-v7-final** (3.46M parameters, ~6 ms per 1 KB note) and **phi-guard-v7-fast** (0.87M, ~1.3 ms; less exact). The example notes are invented; the last part scores the test set in `dataset/`.

Repo layout:

```
models/phi-guard-v7-final-onnx/   models/phi-guard-v7-fast-onnx/   (model.onnx, note.json, score.py, config.json, README.md)
dataset/leak_test_df_doc.csv     (sentence, deidentified, expected_status, ...)
notebooks/phi_guard_v7_demo.ipynb
```

## Setup

In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models").is_dir())   # the repo folder
WHEELS = ROOT / "wheels"   # optional: packages for an offline machine (pip download -d wheels onnxruntime numpy pandas)
cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "onnxruntime", "numpy", "pandas"]
if WHEELS.is_dir():
    cmd += ["--no-index", "--find-links", str(WHEELS)]   # install from wheels/, without the network
subprocess.check_call(cmd)
print("repo:", ROOT)

## Config
Every later cell reads from `CONFIG`. Paths are relative to the repo folder: the models are in `models/`, the test set in `dataset/`.

In [3]:
CONFIG = {
    "models": {"v7-final": ROOT / "models/phi-guard-v7-final-onnx",
               "v7-fast": ROOT / "models/phi-guard-v7-fast-onnx"},
    "threads": 4,                       # onnxruntime CPU threads per model
    "csv_in": ROOT / "dataset/leak_test_df_doc.csv",   # notes to score; None to skip
    "text_column": "deidentified",      # the de-identified text the model reads
    "label_column": "expected_status",  # ground truth (PASS / FAILED); None if there is none
    "csv_out": ROOT / "outputs/v7_scores.csv",   # row, scores, flags and the label (no text)
    "model_for_csv": "v7-final",
}

## Load the models
Each model folder holds `model.onnx`, `note.json`, `score.py` (the scorer, numpy + onnxruntime only), `config.json` and `README.md`.

In [ ]:
import importlib.util
from pathlib import Path

G = {}
for name, folder in CONFIG["models"].items():
    folder = Path(folder)
    spec = importlib.util.spec_from_file_location(f"score_{name}", folder / "score.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    G[name] = mod.PhiGuard(folder, threads=CONFIG["threads"])
    print(f"{name}: loaded from {folder}")
NAMES = mod.NAMES

## Example notes
Invented notes covering each output: clean and leaking de-identified notes, a harmful vs a harmless leak, a PHI-dense note, and raw notes.

In [5]:
NOTES = {
    "masked, clean": "Pt [NAME] seen on [DATE] for chest pain. Troponin negative x2, ECG normal sinus rhythm. Discharged home on aspirin 81 mg daily; follow up with [NAME] in 2 weeks.",
    "masked, patient name leaked": "Pt Maria Delgado seen on [DATE] for chest pain. Troponin negative x2, ECG normal sinus rhythm. Discharged home on aspirin 81 mg daily.",
    "masked, DOB + MRN leaked (harmful)": "Patient [NAME], DOB 03/14/1956, MRN 4471923, admitted on [DATE] with community-acquired pneumonia. Started ceftriaxone and azithromycin.",
    "masked, clinician name only (harmless alone)": "Pt [NAME] seen on [DATE] for knee pain. Dr. Okafor reviewed the MRI and recommended physical therapy twice weekly.",
    "masked, many leaks": "Robert Chen (DOB 7/2/1948, MRN 88213, SSN 512-44-9087) of 41 Birch Lane, Dayton OH, phone 937-555-0142, seen 5/9/2024 at Miami Valley Hospital by Dr. Patel.",
    "raw note with PHI": "John Miller, 64, was seen on 2024-02-11 at St. Luke's for shortness of breath. Contact: 555-201-3344.",
    "raw note, no PHI": "Shortness of breath on exertion, improved with albuterol. Lungs clear on auscultation. Continue current inhaler regimen.",
}

## All outputs, both models

In [6]:
import numpy as np, pandas as pd

def table(g):
    rows = []
    for k, t in NOTES.items():
        p = g.scores(t)
        lv = int((p[2:8] >= 0.5).sum())
        rows.append(dict(note=k, leak=round(p[0], 3), leak_flag=p[0] >= g.threshold, harmful=round(p[1], 3), harmful_flag=p[1] >= g.threshold,
                         level=lv, level_name=NAMES[lv], raw=round(p[8], 3), raw_flag=p[8] >= 0.5))
    return pd.DataFrame(rows).set_index("note")

for name, g in G.items():
    print(name)
    display(table(g))

v7-final


,leak,leak_flag,harmful,harmful_flag,level,level_name,raw,raw_flag
note,,,,,,,,
"masked, clean",0.010,False,0.002,False,0,none,0.000,False
"masked, patient name leaked",1.000,True,0.905,True,2,low,1.000,True
"masked, DOB + MRN leaked (harmful)",1.000,True,1.000,True,2,low,1.000,True
"masked, clinician name only (harmless alone)",0.999,True,0.853,True,1,trace,0.711,True
"masked, many leaks",1.000,True,0.999,True,5,very_high,1.000,True
raw note with PHI,1.000,True,0.972,True,4,high,1.000,True
"raw note, no PHI",0.001,False,0.000,False,0,none,0.000,False


v7-fast


,leak,leak_flag,harmful,harmful_flag,level,level_name,raw,raw_flag
note,,,,,,,,
"masked, clean",0.016,False,0.010,False,0,none,0.0,False
"masked, patient name leaked",1.000,True,0.978,True,2,low,1.0,True
"masked, DOB + MRN leaked (harmful)",1.000,True,0.999,True,2,low,1.0,True
"masked, clinician name only (harmless alone)",1.000,True,0.954,True,1,trace,1.0,True
"masked, many leaks",1.000,True,0.999,True,6,saturated,1.0,True
raw note with PHI,1.000,True,0.994,True,4,high,1.0,True
"raw note, no PHI",0.002,False,0.000,False,0,none,0.0,False


Read it like this:
- On **de-identified** text use `leak` / `harmful` / `level`; `raw` answers a different question (it treats placeholders as fine).
- On **raw** text use `raw` (skip PHI-free notes) and `level` (how much PHI); `leak` assumes masking already ran.
- `harmful` is never above `leak`, and is meant to stay low when only harmless PHI leaked (a clinician name alone, or a date that is not a date of birth). It is not perfect there: in the example above the clinician-only note scores lower than the patient-name note but is still flagged, and on our test set v7-final flags 38% of harmless-alone leaks at 0.5 (v7-fast 21%, phi-guard-v6's leak score 70%).

## v7-final vs v7-fast on the same notes

In [7]:
cmp = pd.concat({n: table(g)[["leak", "harmful", "level", "raw"]] for n, g in G.items()}, axis=1)
cmp

v7-final                       \
                                                 leak harmful level    raw   
note                                                                         
masked, clean                                   0.010   0.002     0  0.000   
masked, patient name leaked                     1.000   0.905     2  1.000   
masked, DOB + MRN leaked (harmful)              1.000   1.000     2  1.000   
masked, clinician name only (harmless alone)    0.999   0.853     1  0.711   
masked, many leaks                              1.000   0.999     5  1.000   
raw note with PHI                               1.000   0.972     4  1.000   
raw note, no PHI                                0.001   0.000     0  0.000   

                                             v7-fast                     
                                                leak harmful level  raw  
note                                                                     
masked, clean                                  0.016   0.010     0  0.0  
masked, patient name leaked                    1.000   0.978     2  1.0  
masked, DOB + MRN leaked (harmful)             1.000   0.999     2  1.0  
masked, clinician name only (harmless alone)   1.000   0.954     1  1.0  
masked, many leaks                             1.000   0.999     6  1.0  
raw note with PHI                              1.000   0.994     4  1.0  
raw note, no PHI                               0.002   0.000     0  0.0

## Richness detail
`score_k` = P(level >= k). The level is how many are >= 0.5: 0 none, 1 trace (under 1 PHI span per 100 words), 2 low (1-2), 3 moderate (2-3), 4 high (3-5), 5 very high (5-8), 6 saturated (8+).

In [8]:
pd.DataFrame({k: G["v7-final"].scores(t)[2:8].round(3) for k, t in NOTES.items()}, index=[f"score_{i}" for i in range(1, 7)]).T

,score_1,score_2,score_3,score_4,score_5,score_6
"masked, clean",0.000,0.000,0.000,0.000,0.000,0.000
"masked, patient name leaked",1.000,0.999,0.013,0.000,0.000,0.000
"masked, DOB + MRN leaked (harmful)",1.000,1.000,0.015,0.000,0.000,0.000
"masked, clinician name only (harmless alone)",0.758,0.000,0.000,0.000,0.000,0.000
"masked, many leaks",1.000,1.000,1.000,1.000,0.998,0.445
raw note with PHI,1.000,1.000,1.000,0.989,0.107,0.002
"raw note, no PHI",0.000,0.000,0.000,0.000,0.000,0.000


## Ranking with the logit
Very confident notes all round to 1.0 as probabilities. To rank notes or compute PR-AUC, use the logit (`g.logits(text)[0]` for the leak output).

In [9]:
pd.Series({k: G["v7-final"].logit(t) for k, t in NOTES.items()}, name="leak logit").sort_values(ascending=False).round(2)

masked, many leaks                              16.07
raw note with PHI                               15.46
masked, DOB + MRN leaked (harmful)              14.62
masked, patient name leaked                     11.93
masked, clinician name only (harmless alone)     6.70
masked, clean                                   -4.62
raw note, no PHI                                -6.97
Name: leak logit, dtype: float64

## Score the test set
Scores `CONFIG["text_column"]` of `CONFIG["csv_in"]` (here `deidentified` of `dataset/leak_test_df_doc.csv`) with every output, and writes one row per note (the same score columns as `python score.py notes.csv out.csv`, no text). With a `label_column`, `flag` (leak score ≥ 0.5 → `FAILED`) is compared with the ground truth.

In [10]:
if CONFIG["csv_in"]:
    g = G[CONFIG["model_for_csv"]]
    df = pd.read_csv(CONFIG["csv_in"], keep_default_na=False)
    P = np.array([g.scores(t) for t in df[CONFIG["text_column"]]])
    lv = (P[:, 2:8] >= 0.5).sum(1)
    out = pd.DataFrame({"row": df.index, "score": P[:, 0], "flag": (P[:, 0] >= g.threshold).astype(int),
                        "harmful_score": P[:, 1], "harmful_flag": (P[:, 1] >= g.threshold).astype(int), "level": lv, "level_name": [NAMES[i] for i in lv]}
                       | {f"score_{k}": P[:, 1 + k] for k in range(1, 7)} | {"raw_score": P[:, 8], "raw_flag": (P[:, 8] >= 0.5).astype(int)})
    label = CONFIG["label_column"]
    if label:
        out[label] = df[label]
        out["predicted"] = np.where(out["flag"] == 1, "FAILED", "PASS")
    CONFIG["csv_out"].parent.mkdir(exist_ok=True)
    out.to_csv(CONFIG["csv_out"], index=False)
    print(f"{len(out):,} notes scored with {CONFIG['model_for_csv']} -> {CONFIG['csv_out']}")
    if label:
        truth, pred = out[label] == "FAILED", out["predicted"] == "FAILED"
        display(pd.crosstab(out[label], out["predicted"], margins=True).rename_axis(index="expected", columns="predicted"))
        print(f"correct {(truth == pred).sum()}/{len(out)} | leaks caught {(truth & pred).sum()}/{truth.sum()} | "
              f"false alarms {(~truth & pred).sum()}/{(~truth).sum()}")
    display(out.head())
else:
    print("set CONFIG['csv_in'] to score your own notes")

789 notes scored with v7-final -> /home/ubuntu/cabir/repo/data-drifting/outputs/v7_scores.csv


predicted,FAILED,PASS,All
expected,,,
FAILED,513,34,547
PASS,11,231,242
All,524,265,789


correct 744/789 | leaks caught 513/547 | false alarms 11/242


,row,score,flag,harmful_score,harmful_flag,level,level_name,score_1,score_2,score_3,score_4,score_5,score_6,raw_score,raw_flag,expected_status,predicted
0,0,0.024819,0,0.024819,0,0,none,0.022083,0.005675,0.000177,2.959218e-05,2.847220e-06,1.751817e-06,0.000089,0,PASS,PASS
1,1,0.017382,0,0.002427,0,0,none,0.009962,0.000589,0.000018,5.964810e-07,5.441200e-07,3.793572e-08,0.000566,0,PASS,PASS
2,2,0.002848,0,0.000272,0,0,none,0.000685,0.000341,0.000002,2.781719e-08,2.781719e-08,5.041373e-09,0.000003,0,PASS,PASS
3,3,0.031067,0,0.007160,0,0,none,0.003216,0.000107,0.000015,2.123416e-06,2.123416e-06,1.418651e-06,0.000077,0,PASS,PASS
4,4,0.014550,0,0.001001,0,0,none,0.000507,0.000030,0.000001,1.142875e-07,1.142875e-07,1.142875e-07,0.000001,0,PASS,PASS


## Classification report
The model's predictions (`predicted`: leak score ≥ 0.5 → `FAILED`) against the ground truth (`expected_status`). `FAILED` (PHI left in the text) is the positive class: **precision** = flagged rows that really leak, **recall** = leaking rows that are flagged, **F1** = their harmonic mean. The table below gives the same for each class, with accuracy and the macro and weighted averages, laid out like scikit-learn's `classification_report`.

In [13]:
def classification_report(y_true, y_pred, labels=("PASS", "FAILED")):
    """Precision, recall, F1 and support per class, accuracy, macro and weighted averages (as sklearn.metrics.classification_report)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rows = {}
    for c in labels:
        tp, fp, fn = ((y_pred == c) & (y_true == c)).sum(), ((y_pred == c) & (y_true != c)).sum(), ((y_pred != c) & (y_true == c)).sum()
        precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        rows[c] = {"precision": precision, "recall": recall, "f1-score": 2 * precision * recall / max(precision + recall, 1e-12),
                   "support": (y_true == c).sum()}
    report = pd.DataFrame(rows).T
    per_class = report.loc[list(labels), ["precision", "recall", "f1-score"]]
    report.loc["accuracy"] = [np.nan, np.nan, (y_true == y_pred).mean(), len(y_true)]
    report.loc["macro avg"] = [*per_class.mean(), len(y_true)]
    report.loc["weighted avg"] = [*np.average(per_class, axis=0, weights=report.loc[list(labels), "support"]), len(y_true)]
    return report.astype({"support": int}).round(4)


if CONFIG["csv_in"] and CONFIG["label_column"]:
    report = classification_report(out[CONFIG["label_column"]], out["predicted"])
    display(pd.Series({"accuracy": report.loc["accuracy", "f1-score"], "precision (FAILED)": report.loc["FAILED", "precision"],
                       "recall (FAILED)": report.loc["FAILED", "recall"], "F1 (FAILED)": report.loc["FAILED", "f1-score"]},
                      name=CONFIG["model_for_csv"]).to_frame())
    display(report.apply(lambda col: col if col.name == "support" else col.map(lambda v: "" if pd.isna(v) else f"{v:.4f}")))
else:
    print("needs CONFIG['csv_in'] and CONFIG['label_column']")

,v7-final
accuracy,0.9430
precision (FAILED),0.9790
recall (FAILED),0.9378
F1 (FAILED),0.9580


,precision,recall,f1-score,support
PASS,0.8717,0.9545,0.9112,242
FAILED,0.9790,0.9378,0.9580,547
accuracy,,,0.9430,789
macro avg,0.9254,0.9462,0.9346,789
weighted avg,0.9461,0.9430,0.9436,789


## Speed
One CPU thread, one 1 KB note, all four outputs (best of 5 x 100 runs).

In [ ]:
import timeit
one_kb = ("Pt [NAME] seen on [DATE] for chest pain; labs stable, meds unchanged. " * 15)[:1024]
for name, folder in CONFIG["models"].items():
    g1 = type(G[name])(folder, threads=1)
    for _ in range(20):
        g1.logits(one_kb)
    ms = min(timeit.timeit(lambda: g1.logits(one_kb), number=100) for _ in range(5)) * 10
    print(f"{name}: {ms:.2f} ms per 1 KB note")

## Limits
- It says whether, and how much, PHI a note holds; it does not find or remove PHI and is not a de-identifier.
- The harmful policy (clinician names and non-birth dates harmless alone, unless 3+ kinds appear together; ages over 89 harmful) is a default.
- v7-fast is about a fifth of v7-final's time; it ranks leaks and richness less exactly. Use v7-final where accuracy matters more than speed.
- Each model folder's README.md has the full test results.